# P85 — Técnicas de factorización matricial para sistemas de recomendación

## 1. Título y paper

**Paper:** *Matrix Factorization Techniques for Recommender Systems*  
**Autoría:** Yehuda Koren, Robert Bell, Chris Volinsky  
**Año y venue:** 2009 · IEEE Computer, 42(8), 30–37  
**Nivel:** L3 · **Motor:** `factorizacion_matricial`  
**Ficha completa:** [`P85_factorizacion_matricial`](../../papers/foundational/P85_factorizacion_matricial/README.md)

**Hito:** El método que ganó el Netflix Prize, explicado con lo que de verdad importa: los sesgos antes que los gustos.

- [doi:10.1109/MC.2009.263](https://doi.org/10.1109/MC.2009.263)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Recomendar exige predecir puntuaciones en una matriz usuario×artículo donde falta el 99 % de las celdas. Los métodos por vecindad escalaban mal y no capturaban estructura latente.
2. Ejecutar una implementación mínima de la propuesta: Aprender un vector de factores latentes por usuario y por artículo, ajustados solo sobre las celdas observadas por descenso de gradiente, con regularización y con términos de sesgo explícitos para usuario y artículo.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P53
- Sarwar et al. (2001), filtrado colaborativo por artículos


## 4. Intuición

Una matriz de usuarios por artículos con el 99 % de celdas vacías. La idea: cada usuario y cada artículo se describen con unos pocos números —factores latentes— que nadie declara y que salen del ajuste. La predicción es el producto escalar de ambos, más los sesgos.


## 5. Concepto mínimo

```text
r̂(u,i) = μ + b_u + b_i + p_u · q_i

    μ    media global
    b_u  ¿este usuario puntúa alto o bajo en general?
    b_i  ¿este artículo gusta a todos o a nadie?
    p_u·q_i  el gusto propiamente dicho

Se ajusta SOLO sobre las celdas observadas, con regularización.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('factorizacion_matricial', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué error da predecir siempre la media?
2. ¿Y añadiendo solo los sesgos?
3. ¿Y con dos factores latentes?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('factorizacion_matricial', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('factorizacion_matricial', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Sobre las celdas que el modelo nunca vio: predecir la media da RMSE **0,7989**; solo los sesgos, **0,5786**; con dos factores latentes, **0,3691**. Los sesgos hacen más de la mitad del camino antes de que aparezca ningún «gusto».


## 10. Comentario pedagógico

Ese es el mensaje práctico del artículo y el que más se salta: modelar primero lo aburrido. Hay usuarios que puntúan alto todo y artículos que gustan a todos, y si no lo separas, tus factores latentes acaban aprendiendo eso en vez de aprender preferencias. Sin las dos líneas base, un RMSE suelto no dice nada.


## 11. Error o anti-patrón deliberado

Anti-patrón: evaluar un recomendador solo por el error de predicción de la nota.


In [ ]:
print('El Netflix Prize se gano optimizando RMSE. Netflix nunca desplego el modelo ganador.')
print('Lo que importa en produccion es el orden de los diez primeros, la diversidad,')
print('la novedad y el coste de servirlo. Acertar la nota es otro objetivo.')

## 12. Corrección

Las líneas base sin las que el número no se puede leer:


In [ ]:
r = run_paper_lab('factorizacion_matricial', seed=7)['result']
print('densidad de la matriz     :', r['densidad'])
print('RMSE prediciendo la media :', r['rmse_prediciendo_la_media'])
print('RMSE solo con sesgos      :', r['rmse_solo_con_sesgos'])
print('RMSE con factores latentes:', r['rmse_con_factores_latentes'])

## 13. Desafío guiado

Mira los vectores de factores de usuario y comprueba si se separan en dos grupos. Nadie le dijo al modelo que existieran.


In [ ]:
r = run_paper_lab('factorizacion_matricial', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa la factorización sobre un conjunto público de valoraciones, con regularización elegida por validación. Compara con las dos líneas base y con un recomendador por popularidad.


## 15. Evidencia de aprendizaje

Guarda la comparación con las dos líneas base y tu explicación de por qué los sesgos van antes que los factores.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P85_factorizacion_matricial/README.md) · evaluación formal: [`assessments/papers/P85_factorizacion_matricial.md`](../../assessments/papers/P85_factorizacion_matricial.md)


## 16. Cierre

Queda un tipo de dato que no se comporta como los demás: el que llega ordenado en el tiempo, donde el futuro no se puede barajar con el pasado.


## 17. Conexión con el siguiente hito

- P05

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
